In [1]:
def simulate_auction(bids_existing, asks_existing, my_side, my_price, my_qty, exit_price, fee_per_unit=0.0):
    """
    Simulate a single-price auction with either a BUY or SELL order from me.

    bids_existing: list of (price, volume)
    asks_existing: list of (price, volume)

    my_side: "buy" or "sell"
    my_price: my limit price
    my_qty: my quantity

    exit_price:
        - if my_side == "buy", this is the later sell price
        - if my_side == "sell", this is the later buyback price

    fee_per_unit: fee paid when closing position

    Returns:
        clearing_price, my_filled_qty, pnl
    """

    # existing orders: me=False, my order: me=True
    all_bids = [(p, v, False) for p, v in bids_existing]
    all_asks = [(p, v, False) for p, v in asks_existing]

    if my_side == "buy":
        all_bids.append((my_price, my_qty, True))
    elif my_side == "sell":
        all_asks.append((my_price, my_qty, True))
    else:
        raise ValueError("my_side must be 'buy' or 'sell'")

    # candidate clearing prices = all prices appearing in book
    candidate_prices = sorted(
        set([p for p, _, _ in all_bids] + [p for p, _, _ in all_asks])
    )

    if not candidate_prices:
        return None, 0, 0.0

    # find clearing price:
    # 1) maximize traded volume
    # 2) if tie, choose higher price
    best_cp = None
    best_volume = -1

    for cp in candidate_prices:
        total_bid_vol = sum(v for p, v, _ in all_bids if p >= cp)
        total_ask_vol = sum(v for p, v, _ in all_asks if p <= cp)
        traded_vol = min(total_bid_vol, total_ask_vol)

        if traded_vol > best_volume or (traded_vol == best_volume and (best_cp is None or cp > best_cp)):
            best_volume = traded_vol
            best_cp = cp

    if best_volume <= 0:
        return best_cp, 0, 0.0

    clearing_price = best_cp
    traded_volume = best_volume

    # allocate fills on MY side only
    my_filled = 0

    if my_side == "buy":
        # eligible bids: price >= clearing price
        # priority: higher price first, then time priority (I'm last at my price)
        eligible_bids = [(p, v, me) for p, v, me in all_bids if p >= clearing_price]
        eligible_bids.sort(key=lambda x: (-x[0], x[2]))  # existing first, me last

        remaining = traded_volume
        for p, v, me in eligible_bids:
            if remaining <= 0:
                break
            fill = min(v, remaining)
            remaining -= fill
            if me:
                my_filled = fill

        pnl = my_filled * (exit_price - clearing_price - fee_per_unit)

    else:  # my_side == "sell"
        # eligible asks: price <= clearing price
        # priority: lower price first, then time priority (I'm last at my price)
        eligible_asks = [(p, v, me) for p, v, me in all_asks if p <= clearing_price]
        eligible_asks.sort(key=lambda x: (x[0], x[2]))  # existing first, me last

        remaining = traded_volume
        for p, v, me in eligible_asks:
            if remaining <= 0:
                break
            fill = min(v, remaining)
            remaining -= fill
            if me:
                my_filled = fill

        pnl = my_filled * (clearing_price - exit_price - fee_per_unit)

    return clearing_price, my_filled, pnl

def find_best_orders(
    bids_existing,
    asks_existing,
    exit_price,
    fee_per_unit,
    price_range,
    volume_range,
    sides=("buy", "sell"),
    top_n=10,
    price_step=1,
    volume_step=1,
    only_filled=True
):
    """
    Search over both side/price/volume and print top N results.
    """

    results = []

    for side in sides:
        for price in range(price_range[0], price_range[1] + 1, price_step):
            for vol in range(volume_range[0], volume_range[1] + 1, volume_step):
                cp, filled, pnl = simulate_auction(
                    bids_existing,
                    asks_existing,
                    side,
                    price,
                    vol,
                    exit_price,
                    fee_per_unit
                )

                if only_filled and filled == 0:
                    continue

                results.append((pnl, side, price, vol, cp, filled))

    results.sort(key=lambda x: x[0], reverse=True)

    print(f"Top {top_n} results:")
    print(f"{'Rank':<5} {'Side':<6} {'Price':<8} {'Qty':<10} {'ClearPx':<10} {'Filled':<10} {'PnL':<12}")
    print("-" * 70)

    for i, (pnl, side, price, vol, cp, filled) in enumerate(results[:top_n], start=1):
        print(f"{i:<5} {side:<6} {price:<8} {vol:<10} {cp:<10} {filled:<10} {pnl:<12.2f}")

    return results[:top_n]

In [5]:
print("=== DRYLAND FLAX ===")
flax_bids = [
    (30, 30000),
    (29, 5000),
    (28, 12000),
    (27, 28000),
]

flax_asks = [
    (28, 40000),
    (31, 20000),
    (32, 20000),
    (33, 30000),
]
find_best_orders(
    flax_bids,
    flax_asks,
    exit_price=30,
    fee_per_unit=0,
    price_range=(25, 35),
    volume_range=(1000, 50000),
    sides=("buy", "sell"),
    top_n=10,
    volume_step=1
)



=== DRYLAND FLAX ===
Top 10 results:
Rank  Side   Price    Qty        ClearPx    Filled     PnL         
----------------------------------------------------------------------
1     buy    30       9999       29         9999       9999.00     
2     buy    31       9999       29         9999       9999.00     
3     buy    32       9999       29         9999       9999.00     
4     buy    33       9999       29         9999       9999.00     
5     buy    34       9999       29         9999       9999.00     
6     buy    35       9999       29         9999       9999.00     
7     buy    29       4999       28         4999       9998.00     
8     buy    30       4999       28         4999       9998.00     
9     buy    30       9998       29         9998       9998.00     
10    buy    31       4999       28         4999       9998.00     


[(9999, 'buy', 30, 9999, 29, 9999),
 (9999, 'buy', 31, 9999, 29, 9999),
 (9999, 'buy', 32, 9999, 29, 9999),
 (9999, 'buy', 33, 9999, 29, 9999),
 (9999, 'buy', 34, 9999, 29, 9999),
 (9999, 'buy', 35, 9999, 29, 9999),
 (9998, 'buy', 29, 4999, 28, 4999),
 (9998, 'buy', 30, 4999, 28, 4999),
 (9998, 'buy', 30, 9998, 29, 9998),
 (9998, 'buy', 31, 4999, 28, 4999)]

In [6]:
print("=== EMBER MUSHROOM ===")
mushroom_bids = [
    (20, 43000),
    (19, 17000),
    (18, 6000),
    (17, 5000),
    (16, 10000),
    (15, 5000),
    (14, 10000),
    (13, 7000),
]

mushroom_asks = [
    (12, 20000),
    (13, 25000),
    (14, 35000),
    (15, 6000),
    (16, 5000),
    (17, 0),
    (18, 10000),
    (19, 12000),
]
find_best_orders(
    mushroom_bids,
    mushroom_asks,
    exit_price=20,
    fee_per_unit=0.10,
    price_range=(10, 21),
    volume_range=(1000, 50000),
    sides=("buy", "sell"),
    top_n=10,
    volume_step=1
)

=== EMBER MUSHROOM ===
Top 10 results:
Rank  Side   Price    Qty        ClearPx    Filled     PnL         
----------------------------------------------------------------------
1     buy    17       19999      16         19999      77996.10    
2     buy    18       19999      16         19999      77996.10    
3     buy    19       19999      16         19999      77996.10    
4     buy    20       19999      16         19999      77996.10    
5     buy    21       19999      16         19999      77996.10    
6     buy    17       19998      16         19998      77992.20    
7     buy    18       19998      16         19998      77992.20    
8     buy    19       19998      16         19998      77992.20    
9     buy    20       19998      16         19998      77992.20    
10    buy    21       19998      16         19998      77992.20    


[(77996.09999999999, 'buy', 17, 19999, 16, 19999),
 (77996.09999999999, 'buy', 18, 19999, 16, 19999),
 (77996.09999999999, 'buy', 19, 19999, 16, 19999),
 (77996.09999999999, 'buy', 20, 19999, 16, 19999),
 (77996.09999999999, 'buy', 21, 19999, 16, 19999),
 (77992.2, 'buy', 17, 19998, 16, 19998),
 (77992.2, 'buy', 18, 19998, 16, 19998),
 (77992.2, 'buy', 19, 19998, 16, 19998),
 (77992.2, 'buy', 20, 19998, 16, 19998),
 (77992.2, 'buy', 21, 19998, 16, 19998)]